# Model Experimentation

### Experiments:
-"distilbert-base-multilingual-cased" (base) -> 20% head tunned
-"distilbert-base-multilingual-cased" (LORA) -> add and train new layer
-"microsoft/mdeberta-v3-base" (base) -> 20% head tunned
-"microsoft/mdeberta-v3-base" (LORA) -> add and train new layer


In [1]:
# ----import tools----

#utilities
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#text processing
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer
)

#LORA Fineting
from peft import LoraConfig, get_peft_model

# Tensorflow and PyTorch
import tensorflow as tf

# CollumnTransformer and Pipeline for preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#evaluation
import evaluate

# MLflow
import mlflow

In [2]:
# Load Data
df = pd.read_csv("../data/processed/cleaned_phishing_email_dataset.csv")
df.head()

,email_text,is_scam
0,check the lowprices on watches and s . ave on ...,1
1,cheating women meet married women who want séx...,1
2,\n BODY { FONT-SIZE: 12px; FONT-FAMILY: Ari...,1
3,empty,1
4,drive for free today to unsubscribe from this ...,1


In [3]:
# Change to HuggingFace Dataset
from datasets import Dataset
dataset = Dataset.from_pandas(df)

In [4]:
dataset

Dataset({
    features: ['email_text', 'is_scam'],
    num_rows: 14621
})

# Preprocess data for distilbert-base-multilingual

In [5]:
# Tokenization for distilbert
distilbert_model_id = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(distilbert_model_id)

def tokenize_function(examples):
    return tokenizer(examples["email_text"],
                      padding="max_length", 
                      truncation=True,
                      max_length=256)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("is_scam", "labels")
tokenized_dataset

Map:   0%|          | 0/14621 [00:00<?, ? examples/s]

Dataset({
    features: ['email_text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 14621
})

In [6]:
# Split the data into train and test sets
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_set = tokenized_dataset["train"]
val_set = tokenized_dataset["test"]


In [7]:
# check tarinning and validation sets
print("Training set:")
print(train_set)
print("\nValidation set:")
print(val_set)

Training set:
Dataset({
    features: ['email_text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 11696
})

Validation set:
Dataset({
    features: ['email_text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2925
})


# Configurate Distilbert-model

In [36]:
distilbert_model_id = "distilbert-base-multilingual-cased"


bert_base = AutoModelForSequenceClassification.from_pretrained(
    distilbert_model_id, num_labels=2)

# freeze the 80% of model layers
all_params = list(bert_base.parameters())
freeze_until = int(0.999 * len(all_params))

for i, param in enumerate(all_params):
    if i < freeze_until:
        param.requires_grad = False
    else:
        param.requires_grad = True

# set base model for training
trainning_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_strategy="best", # save the model at the end of each epoch
    load_best_model_at_end=True, # load the best model at the end of training
    metric_for_best_model='recall' # use recall to evaluate the best model
)

# fit the model
trainer = Trainer(
    model=bert_base,
    args=trainning_args,
    train_dataset=train_set,
    eval_dataset=val_set,
)

trainer.train()



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.691775
1000,0.691395
1500,0.692488
2000,0.691989


TrainOutput(global_step=2193, training_loss=0.6919199973548649, metrics={'train_runtime': 18038.3582, 'train_samples_per_second': 1.945, 'train_steps_per_second': 0.122, 'total_flos': 4648016084041728.0, 'train_loss': 0.6919199973548649, 'epoch': 3.0})

# Fine-tuning DistilBERT with LoRa

In [9]:
distilbert_model_id = "distilbert-base-multilingual-cased"

bert_base = AutoModelForSequenceClassification.from_pretrained(
    distilbert_model_id, num_labels=2)

# apply LORA finetuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

# get the LORA model
bert_lora = get_peft_model(bert_base, lora_config)
bert_lora.print_trainable_parameters()

# set evaluation metric for LORA finetuning
recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return recall.compute(predictions=predictions, references=labels)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 136,213,252 || trainable%: 0.6512


In [10]:
# Set LORA model for training
training_args_lora = TrainingArguments(
    output_dir="./results_lora",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_strategy="best", # save the model at the end of each epoch
    load_best_model_at_end=True, # load the best model at the end of training
    metric_for_best_model='recall' # use recall to evaluate the best model
)

trainer_lora = Trainer(
    model=bert_lora,
    args=training_args_lora,
    train_dataset=train_set,
    eval_dataset=val_set,
    compute_metrics=compute_metrics
)

trainer_lora.train()

/opt/miniconda3/envs/dlenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
500,0.336823
1000,0.176845
1500,0.139558
2000,0.135248


TrainOutput(global_step=2193, training_loss=0.19175748046652347, metrics={'train_runtime': 3166.169, 'train_samples_per_second': 11.082, 'train_steps_per_second': 0.693, 'total_flos': 2371815319633920.0, 'train_loss': 0.19175748046652347, 'epoch': 3.0})

In [11]:
lora_results = trainer_lora.evaluate()

Training Loss,Validation Loss,Step,Recall
0.135248,0.137964,2193,0.947583


In [12]:
print("LORA Finetuning Results:")
print(lora_results)

LORA Finetuning Results:
{'eval_loss': 0.13796386122703552, 'eval_recall': 0.9475833900612661}


### Logging Model to ML Flow

In [22]:
import mlflow
import mlflow.transformers
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://localhost:8080")
mlflow.set_experiment("scam-detector-finetuning")

with mlflow.start_run(run_name="lora_finetuning") as run:
        # 1. Log params
        mlflow.log_params({
        "model_id":      distilbert_model_id,
        "epochs":        3,
        "batch_size":    16,
        "lora_r":        8,
        "lora_alpha":    16,
        "max_length":    256,
        "train_samples": len(train_set),
        "val_samples":   len(val_set),
    })
            
        # 2. Log metrics
        mlflow.log_metrics(lora_results)

        # 3. Log model + tokenizer
        mlflow.transformers.log_model(
        transformers_model={
                    "model":     bert_lora,       # model 
                    "tokenizer": tokenizer # tokenizer 
                },
                artifact_path="model",
                task="text-classification",
            )

# Verifikasi model yang sudah di log
client = MlflowClient(tracking_uri="http://localhost:8080")
experiment = client.get_experiment_by_name("scam-detector-finetuning")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
print(f"Runs in experiment 'scam-detector-finetuning': {[run.info.run_id for run in runs]}")


2026/05/01 17:42:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 17:42:42 INFO mlflow.transformers: Overriding save_pretrained to False for PEFT models, following the Transformers behavior. The PEFT adaptor and config will be saved, but the base model weights will not and reference to the HuggingFace Hub repository will be logged instead.
2026/05/01 17:42:46 INFO mlflow.transformers: Skipping saving pretrained model weights to disk as the save_pretrained argumentis set to False. The reference to the HuggingFace Hub repository distilbert-base-multilingual-cased will be logged instead.
2026/05/01 17:42:47 INFO mlflow.transformers: A local checkpoint path or PEFT model is given as the `transformers_model`. To avoid loading the full model into memory, we don't infer the pip requirement for the model. Instead, we will use the default requirements, but it may not capture all required pip libraries for the model. Consider providing the pip 

🏃 View run lora_finetuning at: http://localhost:8080/#/experiments/1/runs/db87fe9108a944e3a171d2468e2764c6
🧪 View experiment at: http://localhost:8080/#/experiments/1
Runs in experiment 'scam-detector-finetuning': ['db87fe9108a944e3a171d2468e2764c6', '7a2682476fab4ad49d8dac6a1901874a']
